# 04 분석용 오믹스 표 (후보 0)
03의 `files_used.csv`가 가리키는 파일로 분석용 행렬을 만든다.
- RNA: TPM·counts (유전자 × 환자), 유전자 주석
- MAF: 비동의 변이 긴 표 + MAF 보유 환자 목록 (변이 0개 환자도 TMB 0으로 남기기 위함)
- CNV: 유전자 × 환자 복제수, RPPA: 단백 × 환자 + 항체 표
- 커널: `C:\Users\Admin\EXAONEPath\exaonepath-venv`
- 출력: `C:\Users\Admin\EXAONEPath_data\v5\04_omics_tables\`

In [1]:
# 1. 경로와 RNA 파일 형식 확인 (v3에서 재사용한 파일이 GDC STAR counts 형식인지)
from pathlib import Path
import pandas as pd

IN3 = Path(r"C:\Users\Admin\EXAONEPath_data\v5\03_omics_download")
OUT = Path(r"C:\Users\Admin\EXAONEPath_data\v5\04_omics_tables")
OUT.mkdir(parents=True, exist_ok=True)
f = pd.read_csv(IN3 / "files_used.csv")
L = dict(tuple(f.groupby("layer")))
pd.read_csv(L["RNA"].path.iat[0], sep="\t", comment="#", nrows=6)

,gene_id,gene_name,gene_type,unstranded,stranded_first,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded
0,N_unmapped,NaN,NaN,2419725,2419725,2419725,NaN,NaN,NaN
1,N_multimapping,NaN,NaN,5959948,5959948,5959948,NaN,NaN,NaN
2,N_noFeature,NaN,NaN,6722374,39935356,40204496,NaN,NaN,NaN
3,N_ambiguous,NaN,NaN,6858141,1687396,1676767,NaN,NaN,NaN
4,ENSG00000000003.15,TSPAN6,protein_coding,5001,2509,2492,62.5468,17.8238,18.5999
5,ENSG00000000005.6,TNMD,protein_coding,0,0,0,0.0000,0.0000,0.0000


In [2]:
# 2. RNA → TPM·counts 행렬 (Ensembl 유전자 행만; N_unmapped 등 요약 행 제외)
def rna(p):
    d = pd.read_csv(p, sep="\t", comment="#", usecols=["gene_id", "gene_name", "gene_type", "unstranded", "tpm_unstranded"])
    return d[d.gene_id.str.startswith("ENSG")].set_index("gene_id")

rna(L["RNA"].path.iat[0])[["gene_name", "gene_type"]].to_csv(OUT / "genes.csv", encoding="utf-8-sig")
tpm, cnt = {}, {}
for k, r in enumerate(L["RNA"].itertuples(), 1):
    d = rna(r.path)
    tpm[r.patient_id], cnt[r.patient_id] = d.tpm_unstranded, d.unstranded
    if k % 200 == 0: print(k, "/", len(L["RNA"]))
tpm, cnt = pd.DataFrame(tpm).astype("float32"), pd.DataFrame(cnt).astype("int32")
tpm.to_parquet(OUT / "rna_tpm.parquet"); cnt.to_parquet(OUT / "rna_counts.parquet")
print("RNA", tpm.shape, "(유전자 × 환자) | 결측", int(tpm.isna().sum().sum()))

200 / 1018
400 / 1018
600 / 1018
800 / 1018
1000 / 1018
RNA (60660, 1018) (유전자 × 환자) | 결측 0


In [3]:
# 3. MAF → 비동의 변이 표
NONSYN = {"Missense_Mutation", "Nonsense_Mutation", "Frame_Shift_Del", "Frame_Shift_Ins", "In_Frame_Del",
          "In_Frame_Ins", "Splice_Site", "Translation_Start_Site", "Nonstop_Mutation"}
COLS = ["Hugo_Symbol", "Chromosome", "Start_Position", "Variant_Classification", "HGVSp_Short"]
maf = pd.concat([pd.read_csv(r.path, sep="\t", comment="#", low_memory=False, usecols=COLS).assign(patient_id=r.patient_id)
                 for r in L["MAF"].itertuples()])
maf = maf[maf.Variant_Classification.isin(NONSYN)]
maf.to_parquet(OUT / "maf_nonsyn.parquet", index=False)
L["MAF"][["patient_id", "cancer"]].to_csv(OUT / "maf_patients.csv", index=False, encoding="utf-8-sig")
print("MAF 비동의 변이", len(maf), "건 /", maf.patient_id.nunique(), "명 (MAF 보유", len(L["MAF"]), "명)")

MAF 비동의 변이 235440 건 / 987 명 (MAF 보유 994 명)


In [4]:
# 4. CNV, RPPA
cnv = pd.concat({r.patient_id: pd.read_csv(r.path, sep="\t", usecols=["gene_name", "copy_number"])
                 .groupby("gene_name").copy_number.mean() for r in L["CNV"].itertuples()}, axis=1)
cnv.astype("float32").to_parquet(OUT / "cnv_gene.parquet")

rp = pd.concat([pd.read_csv(r.path, sep="\t").assign(patient_id=r.patient_id) for r in L["RPPA"].itertuples()])
rp.drop(columns=["patient_id", "protein_expression"]).drop_duplicates("peptide_target").to_csv(
    OUT / "rppa_antibodies.csv", index=False, encoding="utf-8-sig")
rppa = rp.pivot_table(index="peptide_target", columns="patient_id", values="protein_expression")
rppa.to_parquet(OUT / "rppa.parquet")
print("CNV", cnv.shape, "| RPPA", rppa.shape, "(단백 × 환자) | RPPA 결측 비율", round(float(rppa.isna().mean().mean()), 3))

CNV (59390, 1017) | RPPA (468, 686) (단백 × 환자) | RPPA 결측 비율 0.026


In [5]:
# 5. 요약: 암종별 환자 수
cancer = f.drop_duplicates("patient_id").set_index("patient_id").cancer
pd.DataFrame({"RNA": cancer.reindex(tpm.columns).value_counts(), "MAF": L["MAF"].cancer.value_counts(),
              "CNV": cancer.reindex(cnv.columns).value_counts(), "RPPA": cancer.reindex(rppa.columns).value_counts()})

,RNA,MAF,CNV,RPPA
cancer,,,,
LUAD,517,507,516,361
LUSC,501,487,501,325
